# RoBERTa Sentiment Labeling for Relevant Campaign Comments

This notebook labels sentiment only for rows where:

```text
relevance_manual = relevant
```

Rows marked `irrelevant`, blank, or pending relevance review are not used for sentiment labeling.

Model used:

```text
cardiffnlp/twitter-roberta-base-sentiment-latest
```

Why this model:

- It is trained for social-media style text.
- It supports three sentiment classes: negative, neutral, positive.
- It is suitable for Reddit-like short public comments.

Output files:

```text
malaysia_brand_campaign_reactions_relevant_roberta_sentiment.csv
malaysia_brand_campaign_reactions_full_with_roberta_sentiment.csv
```

Recommended report wording:

> Relevant campaign comments were first selected using manual relevance annotation. Sentiment labels were then pseudo-labeled using a pretrained RoBERTa transformer model for social-media sentiment analysis. Non-relevant and pending comments were excluded from sentiment labeling.

In [ ]:
# Run once if packages are not installed.
%pip install -q pandas openpyxl transformers torch tqdm

In [ ]:
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Mount Google Drive when running in Google Colab.
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
except ModuleNotFoundError:
    IN_COLAB = False
    print("Not running in Google Colab. Google Drive mount skipped.")

if IN_COLAB:
    BASE_DIR = Path("/content/drive/MyDrive/TNL_Data")
else:
    # If running locally, put the CSV in ./TNL_Data or edit INPUT_FILE below.
    BASE_DIR = Path.cwd() / "TNL_Data"

INPUT_FILE = BASE_DIR / "malaysia_brand_campaign_reactions_for_manual_labeling.csv"
OUTPUT_RELEVANT_FILE = BASE_DIR / "malaysia_brand_campaign_reactions_relevant_roberta_sentiment.csv"
OUTPUT_FULL_FILE = BASE_DIR / "malaysia_brand_campaign_reactions_full_with_roberta_sentiment.csv"

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
BATCH_SIZE = 16
MAX_LENGTH = 160

print("Input file:", INPUT_FILE)
print("Relevant output:", OUTPUT_RELEVANT_FILE)
print("Full output:", OUTPUT_FULL_FILE)

## 1. Load Dataset

Make sure your CSV already contains `relevance_manual`, and that relevant rows are marked exactly as `relevant`.

In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}. "
        "Put malaysia_brand_campaign_reactions_for_manual_labeling.csv in TNL_Data, "
        "or edit INPUT_FILE to the correct path."
    )

df = pd.read_csv(INPUT_FILE)
required_columns = {"comment_text", "relevance_manual"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df["relevance_manual"] = df["relevance_manual"].fillna("").astype(str).str.strip().str.lower()
df["comment_text"] = df["comment_text"].fillna("").astype(str)

relevant_mask = df["relevance_manual"].eq("relevant")
relevant_df = df[relevant_mask].copy().reset_index(drop=True)

print("Total rows:", len(df))
print("Relevant rows to label:", len(relevant_df))
print("Rows ignored:", len(df) - len(relevant_df))
print("Relevance distribution:")
display(df["relevance_manual"].replace("", "pending_or_blank").value_counts().to_frame("count"))

if len(relevant_df) == 0:
    raise ValueError("No rows with relevance_manual = relevant. Please mark relevant rows first.")

relevant_df.head(10)


## 2. Load RoBERTa Sentiment Model

The first run may take a few minutes because the model has to download.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

id2label = model.config.id2label
print("Model labels:", id2label)

## 3. Predict Sentiment for Relevant Rows Only

In [ ]:
def normalize_model_label(label):
    label = str(label).lower().strip()
    if "negative" in label or label in {"label_0", "0"}:
        return "negative"
    if "neutral" in label or label in {"label_1", "1"}:
        return "neutral"
    if "positive" in label or label in {"label_2", "2"}:
        return "positive"
    return label


def predict_batch(texts):
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        logits = model(**encoded).logits
        probabilities = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    label_ids = probabilities.argmax(axis=1)
    labels = [normalize_model_label(id2label[int(i)]) for i in label_ids]
    scores = probabilities.max(axis=1)
    return labels, scores, probabilities

texts = relevant_df["comment_text"].tolist()
all_labels = []
all_scores = []
all_probabilities = []

for start in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch_texts = texts[start:start + BATCH_SIZE]
    labels, scores, probabilities = predict_batch(batch_texts)
    all_labels.extend(labels)
    all_scores.extend(scores)
    all_probabilities.extend(probabilities)

prob_array = np.vstack(all_probabilities)

relevant_df["sentiment_roberta"] = all_labels
relevant_df["sentiment_roberta_score"] = np.round(all_scores, 4)
relevant_df["sentiment_model"] = MODEL_NAME
relevant_df["sentiment_labeled_date"] = str(date.today())

# Keep class probabilities for checking uncertain predictions.
for label_id, label_name in id2label.items():
    clean_label = normalize_model_label(label_name)
    relevant_df[f"prob_{clean_label}"] = np.round(prob_array[:, int(label_id)], 4)

# If your project needs a single final sentiment column, use this.
relevant_df["sentiment_label"] = relevant_df["sentiment_roberta"]

print("Predicted sentiment distribution:")
display(relevant_df["sentiment_roberta"].value_counts().reindex(["positive", "neutral", "negative"], fill_value=0).to_frame("count"))

relevant_df[["comment_text", "sentiment_roberta", "sentiment_roberta_score"]].head(20)

## 4. Save Outputs

The relevant-only file is the main file for training/evaluation.

The full file keeps all rows but only fills RoBERTa sentiment columns for `relevant` rows.

In [ ]:
# Save relevant-only sentiment-labeled dataset.
relevant_df.to_csv(OUTPUT_RELEVANT_FILE, index=False, encoding="utf-8-sig")

# Save full dataset, with sentiment columns filled only for relevant rows.
full_df = df.copy()
new_columns = [
    "sentiment_roberta",
    "sentiment_roberta_score",
    "sentiment_model",
    "sentiment_labeled_date",
    "prob_negative",
    "prob_neutral",
    "prob_positive",
    "sentiment_label",
]
for col in new_columns:
    full_df[col] = ""

# Match predictions back to original relevant row order.
relevant_original_indices = df.index[relevant_mask].tolist()
for output_col in new_columns:
    if output_col in relevant_df.columns:
        full_df.loc[relevant_original_indices, output_col] = relevant_df[output_col].values

full_df.to_csv(OUTPUT_FULL_FILE, index=False, encoding="utf-8-sig")

print("Saved relevant sentiment dataset:", OUTPUT_RELEVANT_FILE)
print("Saved full dataset with relevant-only sentiment:", OUTPUT_FULL_FILE)
print("Relevant rows saved:", len(relevant_df))

## 5. Check Low-Confidence Predictions

These are the rows most worth spot-checking manually.

In [ ]:
LOW_CONFIDENCE_THRESHOLD = 0.55
low_confidence_df = relevant_df[relevant_df["sentiment_roberta_score"] < LOW_CONFIDENCE_THRESHOLD].copy()

print(f"Low-confidence rows below {LOW_CONFIDENCE_THRESHOLD}:", len(low_confidence_df))
if len(low_confidence_df) > 0:
    display(low_confidence_df[[
        "comment_text",
        "sentiment_roberta",
        "sentiment_roberta_score",
        "prob_negative",
        "prob_neutral",
        "prob_positive",
    ]].head(50))
else:
    print("No low-confidence rows found.")

## 6. Optional: Balance Check

Your current target is 200 per sentiment class. This check tells you how many more relevant labeled comments are needed.

In [ ]:
TARGET_PER_LABEL = 200
counts = relevant_df["sentiment_roberta"].value_counts().reindex(["positive", "neutral", "negative"], fill_value=0)
display(counts.to_frame("count"))

for label, count in counts.items():
    shortage = TARGET_PER_LABEL - int(count)
    if shortage > 0:
        print(f"Need {shortage} more {label} relevant labeled comments.")
    else:
        print(f"{label}: target reached.")